<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: Forecasting as regression

**[Forecasting as regression](https://learning.nextia-ai.com/courses/forecasting/m05/forecasting-as-regression/)** · Forecasting and Time Series: Predict What Comes Next · Module 5, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn a series into a table of features (lags, rolling means, calendar, price, promotion), train one regression for all products (a global model) and one per product, and add both to the backtest table.

| | |
|---|---|
| **Time** | About 40 minutes, of which about 2 minutes of waiting for the backtests |
| **Needs** | An internet connection for the setup cells. No account and no key. The first cell also installs mlforecast 1.1.0 and LightGBM 4.7.0. |
| **You should know** | Linear regression and features, from the [machine learning course](https://learning.nextia-ai.com/courses/ml/). The backtest table, from [Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m05/forecasting-as-regression/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M05-L01-forecasting-as-regression/forecasting-as-regression-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, lightgbm 4.7.0, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 mlforecast==1.1.0 lightgbm==4.7.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)
import mlforecast; print("mlforecast", mlforecast.__version__)
import lightgbm; print("lightgbm", lightgbm.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A series becomes a table

A regression model needs rows of features and a target. For a forecast, each day becomes a row: the target is the day's sales, and the features are values known **before** that day. A **lag** of 7 is the value 7 days before; a **rolling mean** of a lag is the mean of the values before it.

> **Predict.** For Monday 9 June 2025, which value is `lag7`: the sales of Sunday 8 June or of Monday 2 June? Then run the cell.

In [ ]:
parasol = load_sales(["P025"]).set_index("ds")
t = pd.DataFrame({"y": parasol["y"]})
t["lag7"] = t["y"].shift(7)
t["lag14"] = t["y"].shift(14)
t["mean_of_lag7_7days"] = t["y"].shift(7).rolling(7).mean()
t["weekday"] = t.index.dayofweek
t["on_promotion"] = parasol["on_promotion"]
print(t.loc["2025-06-02":"2025-06-09"].to_string())

> **Check.** You should see 8 rows. On 2025-06-09, `lag7` is 13.0: the value of 2025-06-02, the Monday before.

Every feature in a row must be known when the forecast is made. If you forecast 14 days ahead, a lag of 7 is not yet known for days 8 to 14: either you forecast step by step and use your own forecasts as lags (a **recursive** forecast), or you use only lags of 14 days or more. mlforecast does the first by default.

## 2. One model for all products

A **global model** learns from all 50 products at once: it sees many more rows, and it learns patterns that the products share (the weekday, a promotion). The library mlforecast builds the features and trains any scikit-learn model. The next cell trains a linear regression with lags 7, 14 and 28, two rolling means, the weekday, the month, the day of the year, the price and the promotion flag, at the last cut-off, and shows the first features.

In [ ]:
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from sklearn.linear_model import LinearRegression

sales = load_sales(BACKTEST_PRODUCTS)
EXOG = ["price", "on_promotion"]

def make_model(models):
    return MLForecast(models=models, freq="D", lags=[7, 14, 28],
                      lag_transforms={7: [RollingMean(7), RollingMean(28)]},
                      date_features=["dayofweek", "month", "dayofyear"])

m = make_model({"lr": LinearRegression()})
features = m.preprocess(sales[sales["ds"] <= ORIGINS[-1]][["unique_id", "ds", "y"] + EXOG], static_features=[])
print(features.shape[0], "rows,", features.shape[1] - 3, "features:", [c for c in features.columns if c not in ("unique_id", "ds", "y")])

> **Check.** You should see 34000 rows and 10 features: `price`, `on_promotion`, `lag7`, `lag14`, `lag28`, two rolling means, `dayofweek`, `month` and `dayofyear`.

## 3. Global and per-product in the backtest

The next cell wraps the model as a forecaster for `backtest.py` and runs it at the 13 cut-offs. Then it does the same with **one model per product**: 50 small regressions at every cut-off.

> **Predict.** Which will be better: one model that sees 50 products, or 50 models that each see one? Then run the cell (about 2 minutes).

In [ ]:
def global_lr(train, h, future):
    m = make_model({"lr": LinearRegression()})
    m.fit(train[["unique_id", "ds", "y"] + EXOG], static_features=[])
    p = m.predict(h, X_df=future[["unique_id", "ds"] + EXOG])
    return p.rename(columns={"lr": "yhat"}).assign(yhat=lambda d: d["yhat"].clip(lower=0))

def per_product_lr(train, h, future):
    parts = [global_lr(g, h, future[future["unique_id"] == uid]) for uid, g in train.groupby("unique_id")]
    return pd.concat(parts, ignore_index=True)

fc_global = bt.backtest(sales, global_lr, h=14, origins=ORIGINS)
fc_local = bt.backtest(sales, per_product_lr, h=14, origins=ORIGINS)
rows = [bt.score(fc_global, sales, "linear regression, one global model", season=7, cost=bt.stock_cost(PRICES)),
        bt.score(fc_local, sales, "linear regression, one model per product", season=7, cost=bt.stock_cost(PRICES))]
table = bt.show(rows)

> **Check.** You should see the per-product models first, with MASE 0.849 and WAPE % 42.2, then the global model with MASE 0.865 and WAPE % 42.6.

Here the 50 small models are slightly better, and they take more than 30 times longer to fit (about 11 seconds against 0.3). Each product has two years of history, which is enough for its own model. (The course's backtest table also gives both models three holiday flags from `calendar.csv`; its rows are a little different from these.) Compare both with ETS from Module 4 (WAPE 41.3%): a linear model with good features is a serious forecaster, but not a magic one.

## 4. Your turn: what did the promotion add?

> **Your turn.** In `fc_global`, keep the rows where the product was on promotion (join `sales[["unique_id", "ds", "on_promotion"]]` on `unique_id` and `ds`). Compute the bias in % of those rows: `100 * (forecast sum - real sum) / real sum`, rounded to 1 decimal. Put it in `promo_bias` as a Python float. Then run the check cell.

In [ ]:
p = fc_global.merge(sales[["unique_id", "ds", "on_promotion"]], on=["unique_id", "ds"])
p = p[p["on_promotion"] == 1]
promo_bias = round(float(100 * (p["yhat"].sum() - p["y"].sum()) / p["y"].sum()), 1)
print(promo_bias, "% on", len(p), "promotion days")

In [ ]:
expect('promo_bias', promo_bias, 6.8)

Over the 154 promotion days, the global model is 6.8% too high. A linear model adds the same number of units for every promotion, for a slow product and for a fast one, but a promotion **multiplies** sales: a fixed number cannot be right for both. The next lesson's gradient boosting can learn this kind of interaction; a linear model needs you to build it as a feature.

## 5. Failure case: a feature from the future

Tomás adds a rolling mean of the last 7 days **including the day itself**: it looks harmless. The next cell compares it with the honest model at the last cut-off.

In [ ]:
def with_leak(train, h, future, leak):
    full = sales.sort_values(["unique_id", "ds"]).copy()
    shift = 0 if leak else 14
    full["recent_mean"] = full.groupby("unique_id")["y"].transform(lambda s: s.shift(shift).rolling(7, min_periods=1).mean())
    full["dow"] = full["ds"].dt.dayofweek
    tr = full[full["ds"].isin(train["ds"].unique())].dropna()
    te = full.merge(future[["unique_id", "ds"]], on=["unique_id", "ds"])
    model = LinearRegression().fit(tr[["recent_mean", "dow", "on_promotion"]], tr["y"])
    return te[["unique_id", "ds"]].assign(yhat=model.predict(te[["recent_mean", "dow", "on_promotion"]]))

for leak in (True, False):
    f = bt.backtest(sales, lambda tr, h, fu: with_leak(tr, h, fu, leak), h=14, origins=ORIGINS[-1:])
    print("rolling mean including the day itself" if leak else "rolling mean of lags 14 and more  ",
          "WAPE %", bt.score(f, sales, "", season=7)["WAPE %"])

> **Check.** You should see a WAPE of 31.6% with the leak and 48.4% without it (at the last cut-off only).

The leaky model looks much better, because each day's feature contains that day's sales. On a real Monday, Kenji does not know the sales of the next two weeks. **Every feature must be computed only from data before the cut-off.** mlforecast computes its lags and rolling means this way; your own pandas code must do the same.

## Next

You added two regression rows to the backtest table. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m05/forecasting-as-regression/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Gradient boosting with LightGBM](https://learning.nextia-ai.com/courses/forecasting/m05/gradient-boosting-with-lightgbm/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m05/forecasting-as-regression/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.